# Remote .cz access

## 1. Setup the server

### Serving `.cz` Files for Remote Access
 
[cytozip](https://github.com/DingWB/cytozip) supports reading `.cz` files directly over HTTP without downloading the whole file. To make this work, the server hosting your `.cz` files must satisfy two requirements:
 
1. **HTTP Range requests** — so clients can fetch only the byte ranges they need instead of the entire file.
2. **CORS** — required only when the file is read from a **browser** (e.g. a JavaScript/WebAssembly viewer). Pure Python clients (`requests`, `fsspec`, etc.) are **not** subject to CORS and only need Range support.
> **Key detail:** Most static file servers and object stores already support Range requests out of the box. The part you usually have to configure by hand is CORS — and it must both **expose the Range-related response headers** (`Content-Range`, `Accept-Ranges`, `Content-Length`) *and* **answer the `OPTIONS` preflight request**. If either is missing, the browser may receive the bytes but fail to read the offsets, or the request gets blocked before it starts.
 
---
 
#### Nginx
 
```nginx
server {
    listen 443 ssl;
    server_name your-domain.com;
    root /path/to/cz_data;
 
    location ~ \.cz$ {
        # Handle the CORS preflight request
        if ($request_method = OPTIONS) {
            add_header Access-Control-Allow-Origin  "*";
            add_header Access-Control-Allow-Methods "GET, HEAD, OPTIONS";
            add_header Access-Control-Allow-Headers "Range";
            add_header Access-Control-Max-Age       86400;
            add_header Content-Length               0;
            return 204;
        }
 
        # Use `always` so CORS headers are attached to 206 and error responses too
        add_header Access-Control-Allow-Origin   "*" always;
        add_header Access-Control-Expose-Headers "Content-Length, Content-Range, Accept-Ranges" always;
 
        # Range support is enabled by default (Accept-Ranges: bytes is returned automatically)
    }
}
```
 
**Gotchas:**
 
- `add_header` directives declared *outside* an `if` block are **not** inherited by the `204` response from `return`, so the preflight headers must live **inside** the `if` block (as shown above).
- Adding `always` ensures the headers also appear on `206 Partial Content` and error responses.
---
 
#### Apache
 
Requires `mod_headers` and `mod_rewrite`. Place this in your virtual host config or an `.htaccess` file:
 
```apache
<Directory "/home/www/ftp">
    Options Indexes FollowSymLinks
    AllowOverride All
    Require all granted

    Header always set Access-Control-Allow-Origin  "*"
    Header always set Access-Control-Allow-Methods "GET, HEAD, OPTIONS"
    Header always set Access-Control-Allow-Headers "Range, If-Range"
    Header always set Access-Control-Expose-Headers "Content-Length, Content-Range, Accept-Ranges, ETag, Last-Modified"
    Header always set Access-Control-Max-Age "86400"
</Directory>

<Location "/ftp/">
    Header always set Access-Control-Allow-Origin  "*"
    Header always set Access-Control-Allow-Methods "GET, HEAD, OPTIONS"
    Header always set Access-Control-Allow-Headers "Range, If-Range"
    Header always set Access-Control-Expose-Headers "Content-Length, Content-Range, Accept-Ranges, ETag, Last-Modified"
    Header always set Access-Control-Max-Age "86400"
</Location>
```
 
Apache's core supports Range requests by default; no extra module is needed for that.
 
---
 
#### Object Storage (S3 / GCS)
 
Object stores support Range requests natively — you only need to configure CORS.
 
##### Amazon S3
 
Set the bucket's CORS configuration:
 
```json
[
  {
    "AllowedOrigins": ["*"],
    "AllowedMethods": ["GET", "HEAD"],
    "AllowedHeaders": ["Range"],
    "ExposeHeaders": ["Content-Length", "Content-Range", "Accept-Ranges", "ETag"]
  }
]
```
 
##### Google Cloud Storage
 
Create a `cors.json`:
 
```json
[
  {
    "origin": ["*"],
    "method": ["GET", "HEAD"],
    "responseHeader": ["Content-Length", "Content-Range", "Accept-Ranges", "ETag"],
    "maxAgeSeconds": 86400
  }
]
```
 
Apply it:
 
```bash
gcloud storage buckets update gs://your-bucket --cors-file=cors.json
```
 
---
 
#### Testing Your Configuration
 
> **Do not test with `python -m http.server`** — it does **not** support Range requests and will return the full file with a `200` status. Use [`RangeHTTPServer`](https://pypi.org/project/rangehttpserver/) (`pip install rangehttpserver`, then `python -m RangeHTTPServer`) or test against your real server.
 
**1. Verify Range requests** — should return `206 Partial Content` with a `Content-Range` header:
 
```bash
curl -I -H "Range: bytes=0-99" https://your-domain/data.cz
```
 
**2. Verify the CORS preflight** — should return `Access-Control-Allow-*` headers:
 
```bash
curl -I -X OPTIONS \
  -H "Origin: https://example.com" \
  -H "Access-Control-Request-Method: GET" \
  -H "Access-Control-Request-Headers: Range" \
  https://your-domain/data.cz
```
 
**3. Verify a real cross-origin Range request** — confirm the `206` carries both the CORS header and the exposed headers:
 
```bash
curl -I -H "Origin: https://example.com" -H "Range: bytes=0-99" https://your-domain/data.cz
```
 
---
 
> Note: Python-Only Access
 
> - If your `.cz` files are only ever read from Python (via `requests`, `fsspec`, `HTTPFileSystem`, etc.), **CORS is not required** — it is a browser-only security mechanism. In that case you only need to ensure Range requests work, which nearly all static servers and object stores do by default.

## 2. View / query a remote `.cz` (no download)

cytozip reads `.cz` files directly over HTTP Range requests when they
carry a chunk index. Below we query a remote `.cz` hosted on figshare —
only the needed chunks are fetched on-demand.


In [1]:
! czip header -I https://neomorph.salk.edu/ftp/bican/UWA7648_CX1819_NAC_1_P10-1-K18-A10.cz

magic  :  b'CZIP'
version  :  0.36
total_size  :  22810155
message  :  hg38_with_chrL.allc.cz
formats  :  ['B', 'B']
columns  :  ['mc', 'cov']
sort_col  :  None
delta_cols  :  []
chunk_dims  :  ['chrom']
header_size  :  61


In [2]:
# view a cz file (no coordinates were stored) alone
! czip view -I https://neomorph.salk.edu/ftp/bican/UWA7648_CX1819_NAC_1_P10-1-K18-A10.cz --show_dims 0 | head

chrom	mc	cov
chr1	0	0
chr1	0	0
chr1	0	0
chr1	0	0
chr1	0	0
chr1	0	0
chr1	0	0
chr1	0	0
chr1	0	0


In [3]:
# query remote .cz file with local reference .cz:
! time czip query -I https://neomorph.salk.edu/ftp/bican/UWA7648_CX1819_NAC_1_P10-1-K18-A10.cz \
    -r ~/Ref/hg38/hg38_with_chrL.allc.cz -K chr9 \
    -s 3000294 -e 3005294 | head -n 5

chrom	pos	strand	context	mc	cov
chr9	3000294	+	CAA	0	0
chr9	3000297	-	CTT	0	0
chr9	3000299	-	CAC	0	0
chr9	3000301	+	CAT	0	0

real	0m6.022s
user	0m0.258s
sys	0m0.112s


In [ ]:
# or both the cz and the reference can be accessed via HTTP(S) URLs, without downloading any files locally:
! time czip query -I https://neomorph.salk.edu/ftp/bican/UWA7648_CX1819_NAC_1_P10-1-K18-A10.cz \
    -r https://neomorph.salk.edu/ftp/cz/hg38_with_chrL.allc.cz \
    -K chr9 -s 3000294 -e 3005294 | head -n 5
# query 5000 bp of regions containing 1759 records

chrom	pos	strand	context	mc	cov
chr9	3000294	+	CAA	0	0
chr9	3000297	-	CTT	0	0
chr9	3000299	-	CAC	0	0
chr9	3000301	+	CAT	0	0

real	0m2.840s
user	0m0.257s
sys	0m0.095s


## 3. Opening a remote .cz file in Python

There are two ways to open a remote `.cz` file:

**Method 1: `Reader.from_url()` (recommended)**
```python
reader = cytozip.Reader.from_url(url, cache_size=2*1024*1024)
```

**Method 2: Pass URL directly to `Reader()` (auto-detected)**
```python
reader = cytozip.Reader(url)
```

Both methods detect `http://` or `https://` prefixes and automatically use the `RemoteFile` backend.

In [ ]:
import cytozip
from cytozip.cz import RemoteFile
print(f"cytozip version: {cytozip.__version__}")

### Example: Open and inspect a remote file

Replace `url` below with the URL of your `.cz` file:

In [ ]:
# Uncomment and replace with your URL:
# url = "https://your-server.com/data/sample.cz"
# reader = cytozip.Reader.from_url(url)
# reader.print_header()
print("Replace the URL above with a real .cz file URL to test remote reading.")

## 4. Querying remote files in Python

Once opened, all Reader methods work identically on remote and local files:

In [34]:
import cytozip as czip
url = "https://neomorph.salk.edu/ftp/bican/FC_M_P12b_3C_2-5-M17-N10.with_pos.cz"

# Method 1: Using Reader.from_url (recommended)
reader = czip.Reader.from_url(url)

# Method 2: Pass URL directly to Reader (auto-detected)
# reader = czip.Reader(url)

# Once opened, all Reader methods work the same as local files:
reader.print_header()
reader.summary_chunks()
for record in reader.fetch(("chr1",)):
    print(record)
    break
for record in reader.query(chunk_key="chr9",start=3000294,end=3000472,printout=False):
    print(record)
reader.close()

print("Remote reading requires an HTTP server supporting Range requests.")
print("The .cz file should include a chunk index for optimal performance.")

magic  :  b'CZIP'
version  :  0.3
total_size  :  43494399
message  :  
formats  :  ['Q', 'B', 'B']
columns  :  ['pos', 'mc', 'cov']
sort_col  :  0
delta_cols  :  [0]
chunk_dims  :  ['chrom']
header_size  :  47
chrom	chunk_start_offset	chunk_size	chunk_tail_offset	chunk_nblocks	chunk_nrows
chr1	47	3365956	3367784	110	2871952
chr10	3367784	2217013	5585987	73	1893443
chr11	5585987	2063328	7650441	69	1793491
chr12	7650441	2039078	9690613	67	1748109
chr13	9690613	2035966	11727673	67	1740746
chr14	11727673	2070897	13799680	68	1767574
chr15	13799680	1745220	15545850	58	1496229
chr16	15545850	1662163	17208899	54	1414646
chr17	17208899	1593579	18803348	53	1376492
chr18	18803348	1525689	20329859	50	1302683
chr19	20329859	998682	21329091	33	862401
chr1_GL456210_random	21329091	4156	21333300	1	3630
chr1_GL456211_random	21333300	6061	21339414	1	5213
chr1_GL456212_random	21339414	4135	21343602	1	3537
chr1_GL456213_random	21343602	66	21343721	1	22
chr1_GL456221_random	21343721	5424	21349198	1	4668
ch

## 5. Read remote cz file from figshare

In [24]:
import os
import requests
url="https://figshare.com/ndownloader/files/64046767" 
# FC_M_P12b_3C_2-5-M17-N10.cz
session = requests.Session()
session.headers.update({
    "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
                    "AppleWebKit/537.36 (KHTML, like Gecko) "
                    "Chrome/120.0.0.0 Safari/537.36",
    "Referer": "https://figshare.com/",
    "Accept": "*/*",
})
session.get("https://figshare.com")  # acquire cookies
reader = czip.Reader.from_url(url,session=session)
print(reader._is_remote)
print(reader.header)
print(reader.summary_chunks())

# query with reference
for record in reader.query(chunk_key="chr9",start=3000294,end=3000472,
                           reference="output/mm10_with_chrL.allc.cz",
                           # reference="https://neomorph.salk.edu/ftp/bican/mm10_with_chrL.allc.cz", # reference file could also be stored on a remote server
                           printout=False):
    print(record) # list

True
{'magic': b'CZIP', 'version': 0.3, 'total_size': 29849594, 'message': 'mm10_with_chrL.allc.cz', 'formats': ['B', 'B'], 'columns': ['mc', 'cov'], 'sort_col': None, 'delta_cols': [], 'chunk_dims': ['chrom'], 'header_size': 61}
chrom	chunk_start_offset	chunk_size	chunk_tail_offset	chunk_nblocks	chunk_nrows
chr1	61	2275394	2280300	603	78962721
chr10	2280300	1505619	3789157	402	52609184
chr11	3789157	1426655	5219010	397	52027265
chr12	5219010	1386532	6608548	373	48799752
chr13	6608548	1384955	7996501	372	48750883
chr14	7996501	1405739	9405318	382	49987736
chr15	9405318	1192458	10600382	323	42230765
chr16	10600382	1123440	11726220	297	38899643
chr17	11726220	1093682	12822316	299	39153472
chr18	12822316	1035563	13860117	277	36239315
chr19	13860117	685988	14547647	190	24870223
chr1_GL456210_random	14547647	2658	14550350	1	74695
chr1_GL456211_random	14550350	4011	14554406	1	106168
chr1_GL456212_random	14554406	2647	14557098	1	67179
chr1_GL456213_random	14557098	86	14557229	1	17318
chr1_GL4